# Cleaning and split

Applies the cleaning decisions from `00_eda.ipynb` (section 10) to both Kaggle files,
splits the labeled data 80/20 and saves the result to `data/processed/`.

**Names used in the code**

| Name | Data | `TARGET` |
| --- | --- | --- |
| `labeled` | `application_train.csv`: all 307,511 clients with known outcome | yes |
| `kaggle_test` | `application_test.csv`: 48,744 clients, outcome known only to Kaggle | no |
| `train` / `test` | the two parts of `labeled` after the 80/20 split (section 8) | yes |

Only steps that learn nothing from the data happen here: every row is transformed on its own, with fixed rules.
Imputation, log transform, scaling and encoding learn values from the data (medians, means, categories)
and are therefore fitted later, inside each model's pipeline, on the training folds only.

1. Load
2. Placeholder values
3. Time columns in years
4. Ratio features
5. Removed columns
6. Text columns as categories
7. Cleaning both files
8. Stratified 80/20 split
9. Save
10. Summary

Imports, paths and the random seed used for the split.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")
SEED = 69

## 1. Load

Load both files with `SK_ID_CURR` as the row index, so the ID is no longer a column.

In [2]:
labeled = pd.read_csv(RAW / "application_train.csv", index_col="SK_ID_CURR")
kaggle_test = pd.read_csv(RAW / "application_test.csv", index_col="SK_ID_CURR")
print("labeled:    ", labeled.shape, "| index unique:", labeled.index.is_unique)
print("kaggle_test:", kaggle_test.shape, "| index unique:", kaggle_test.index.is_unique)

labeled:     (307511, 121) | index unique: True
kaggle_test: (48744, 120) | index unique: True


## 2. Placeholder values

- `DAYS_EMPLOYED = 365243` stands for "pensioner or unemployed", not for 1,000 years of work. It becomes `NaN`, and the new 0/1 column `NOT_EMPLOYED` keeps the information.
- `CODE_GENDER = 'XNA'` and `NAME_FAMILY_STATUS = 'Unknown'` mean "not recorded" and become `NaN`.
- `ORGANIZATION_TYPE = 'XNA'` stays unchanged as its own category ("no employer").

The function. `df.copy()` leaves the input unchanged.

In [3]:
def fix_placeholders(df):
    df = df.copy()
    df["NOT_EMPLOYED"] = (df["DAYS_EMPLOYED"] == 365243).astype(int)
    df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].replace(365243, np.nan)
    df["CODE_GENDER"] = df["CODE_GENDER"].replace("XNA", np.nan)
    df["NAME_FAMILY_STATUS"] = df["NAME_FAMILY_STATUS"].replace("Unknown", np.nan)
    return df

Check on `labeled`: counts before and after.

In [4]:
fixed = fix_placeholders(labeled)
print("DAYS_EMPLOYED = 365243:  before", (labeled["DAYS_EMPLOYED"] == 365243).sum(), "| after", (fixed["DAYS_EMPLOYED"] == 365243).sum())
print("NOT_EMPLOYED = 1:       ", fixed["NOT_EMPLOYED"].sum())
print("DAYS_EMPLOYED missing:   before", labeled["DAYS_EMPLOYED"].isna().sum(), "| after", fixed["DAYS_EMPLOYED"].isna().sum())
print("CODE_GENDER:            ", fixed["CODE_GENDER"].value_counts(dropna=False).to_dict())
print("NAME_FAMILY_STATUS NaN: ", fixed["NAME_FAMILY_STATUS"].isna().sum())
print("ORGANIZATION_TYPE 'XNA':", (fixed["ORGANIZATION_TYPE"] == "XNA").sum())

DAYS_EMPLOYED = 365243:  before 55374 | after 0
NOT_EMPLOYED = 1:        55374
DAYS_EMPLOYED missing:   before 0 | after 55374
CODE_GENDER:             {'F': 202448, 'M': 105059, nan: 4}
NAME_FAMILY_STATUS NaN:  2
ORGANIZATION_TYPE 'XNA': 55374


## 3. Time columns in years

The five `DAYS_*` columns count days before the application, so they are negative.
They become positive years and are renamed to `YEARS_*`; e.g. `DAYS_BIRTH = −12,000` → `YEARS_BIRTH = 32.9`.
This step needs the placeholder fix first: afterwards no `DAYS_*` value is positive any more, which the function checks.

The function.

In [5]:
def days_to_years(df):
    df = df.copy()
    days = [c for c in df.columns if c.startswith("DAYS_")]
    assert (df[days].fillna(0) <= 0).all().all(), "positive DAYS_ value: placeholder not fixed?"
    for col in days:
        df[col.replace("DAYS_", "YEARS_")] = df[col].abs() / 365.25
    return df.drop(columns=days)

Check: range of the new columns.

In [6]:
years = days_to_years(fixed)
year_cols = [c.replace("DAYS_", "YEARS_") for c in labeled.columns if c.startswith("DAYS_")]
years[year_cols].describe().T[["count", "min", "mean", "max"]].round(1)

,count,min,mean,max
YEARS_BIRTH,307511.0,20.5,43.9,69.1
YEARS_EMPLOYED,252137.0,0.0,6.5,49.0
YEARS_REGISTRATION,307511.0,0.0,13.7,67.5
YEARS_ID_PUBLISH,307511.0,0.0,8.2,19.7
YEARS_LAST_PHONE_CHANGE,307510.0,0.0,2.6,11.8


## 4. Ratio features

Three ratios a lender looks at:
- `CREDIT_INCOME_RATIO` = credit amount / yearly income
- `ANNUITY_INCOME_RATIO` = annuity (regular payment) / yearly income
- `CREDIT_ANNUITY_RATIO` = credit amount / annuity ≈ number of payments

The function.

In [7]:
def add_ratios(df):
    df = df.copy()
    df["CREDIT_INCOME_RATIO"] = df["AMT_CREDIT"] / df["AMT_INCOME_TOTAL"]
    df["ANNUITY_INCOME_RATIO"] = df["AMT_ANNUITY"] / df["AMT_INCOME_TOTAL"]
    df["CREDIT_ANNUITY_RATIO"] = df["AMT_CREDIT"] / df["AMT_ANNUITY"]
    return df

Check: no division by zero (infinite values), and the number of missing values (from missing `AMT_ANNUITY`).

In [8]:
RATIOS = ["CREDIT_INCOME_RATIO", "ANNUITY_INCOME_RATIO", "CREDIT_ANNUITY_RATIO"]
ratios = add_ratios(years)
print("infinite values:", np.isinf(ratios[RATIOS]).sum().sum())
ratios[RATIOS].describe().T[["count", "min", "50%", "max"]].round(2)

infinite values: 0


,count,min,50%,max
CREDIT_INCOME_RATIO,307511.0,0.00,3.27,84.74
ANNUITY_INCOME_RATIO,307499.0,0.00,0.16,1.88
CREDIT_ANNUITY_RATIO,307499.0,8.04,20.00,45.31


## 5. Removed columns

- **10 near-constant columns** (one value in more than 99.9% of the rows): `FLAG_MOBIL` and 9 document flags.
- **28 duplicate building columns:** the `_MODE` and `_MEDI` versions of the 14 building measures that also exist as `_AVG`. The 5 columns that exist only as `_MODE` stay.

The lists, written out so it is visible exactly what is removed.

In [9]:
NEAR_CONSTANT = ["FLAG_MOBIL", "FLAG_DOCUMENT_2", "FLAG_DOCUMENT_4", "FLAG_DOCUMENT_7", "FLAG_DOCUMENT_10",
                 "FLAG_DOCUMENT_12", "FLAG_DOCUMENT_17", "FLAG_DOCUMENT_19", "FLAG_DOCUMENT_20", "FLAG_DOCUMENT_21"]

BUILDING_MEASURES = ["APARTMENTS", "BASEMENTAREA", "YEARS_BEGINEXPLUATATION", "YEARS_BUILD", "COMMONAREA",
                     "ELEVATORS", "ENTRANCES", "FLOORSMAX", "FLOORSMIN", "LANDAREA", "LIVINGAPARTMENTS",
                     "LIVINGAREA", "NONLIVINGAPARTMENTS", "NONLIVINGAREA"]
BUILDING_DUPLICATES = [f"{m}_{v}" for m in BUILDING_MEASURES for v in ("MODE", "MEDI")]

print(len(NEAR_CONSTANT), "near-constant +", len(BUILDING_DUPLICATES), "building duplicates =",
      len(NEAR_CONSTANT) + len(BUILDING_DUPLICATES), "columns")

10 near-constant + 28 building duplicates = 38 columns


Check that the near-constant columns really are near-constant in `labeled`.

In [10]:
top_share = {c: labeled[c].value_counts(normalize=True).iloc[0] for c in NEAR_CONSTANT}
pd.Series(top_share, name="share of the most frequent value").round(5)

FLAG_MOBIL          1.00000
FLAG_DOCUMENT_2     0.99996
FLAG_DOCUMENT_4     0.99992
FLAG_DOCUMENT_7     0.99981
FLAG_DOCUMENT_10    0.99998
FLAG_DOCUMENT_12    0.99999
FLAG_DOCUMENT_17    0.99973
FLAG_DOCUMENT_19    0.99940
FLAG_DOCUMENT_20    0.99949
FLAG_DOCUMENT_21    0.99967
Name: share of the most frequent value, dtype: float64

The function.

In [11]:
def drop_columns(df):
    return df.drop(columns=NEAR_CONSTANT + BUILDING_DUPLICATES)

## 6. Text columns as categories

The 16 text columns get the pandas type `category`. It stores each distinct value once (less memory),
keeps missing values as `NaN` when saved to a file (plain text columns come back as `None`),
and lets gradient boosting recognise these columns as categorical directly.

The function.

In [12]:
def text_to_category(df):
    df = df.copy()
    text_cols = df.select_dtypes("object").columns
    df[text_cols] = df[text_cols].astype("category")
    return df

## 7. Cleaning both files

`clean` runs the five steps in this order. `labeled` and `kaggle_test` are cleaned separately with the same function,
so the model can later predict on `kaggle_test`.

The complete cleaning function, applied to both files.

In [13]:
def clean(df):
    return text_to_category(drop_columns(add_ratios(days_to_years(fix_placeholders(df)))))


labeled_clean = clean(labeled)
kaggle_test_clean = clean(kaggle_test)

features = labeled_clean.columns.drop("TARGET")
print("features:", len(features))
print("labeled_clean:", labeled_clean.shape, "| kaggle_test_clean:", kaggle_test_clean.shape)
print("same feature columns in the same order:", list(features) == list(kaggle_test_clean.columns))

features: 86
labeled_clean: (307511, 87) | kaggle_test_clean: (48744, 86)
same feature columns in the same order: True


Overview of the cleaned feature set: number of columns per data type.

In [14]:
labeled_clean[features].dtypes.astype(str).value_counts()

float64     43
int64       27
category    16
Name: count, dtype: int64

## 8. Stratified 80/20 split

`labeled_clean` is split into our own training set `train` (80%) and test set `test` (20%). The test set comes from the labeled data, so its outcomes are known and every metric can be computed on it.
- **Stratified:** both parts keep the 8.1% default rate.
- **Fixed seed (`SEED = 69`):** the split is the same every time and on every computer.
- **Test set:** used once, for the final evaluation. All tuning happens with cross-validation inside the training set.

The split is stored as a list of client IDs, so every model in the project uses exactly the same clients.

The split.

In [15]:
train_ids, test_ids = train_test_split(
    labeled_clean.index, test_size=0.2, stratify=labeled_clean["TARGET"], random_state=SEED
)
split = pd.Series("train", index=labeled_clean.index, name="split")
split[test_ids] = "test"
split.value_counts()

split
train    246008
test      61503
Name: count, dtype: int64

Check: default rate in each part, and no client in both parts.

In [16]:
print(labeled_clean.groupby(split)["TARGET"].agg(["size", "mean"]).round(4))
print("clients in both parts:", len(set(train_ids) & set(test_ids)))

         size    mean
split                
test    61503  0.0807
train  246008  0.0807


clients in both parts: 0


## 9. Save

| File | Content |
| --- | --- |
| `labeled_clean.parquet` | `labeled_clean`: 307,511 clients with `TARGET` |
| `kaggle_test_clean.parquet` | `kaggle_test_clean`: 48,744 clients without `TARGET` |
| `split.csv` | `SK_ID_CURR` → `train` or `test` |

Parquet keeps the data types (unlike CSV) and loads much faster.

Write the three files.

In [17]:
PROCESSED.mkdir(parents=True, exist_ok=True)
labeled_clean.to_parquet(PROCESSED / "labeled_clean.parquet")
kaggle_test_clean.to_parquet(PROCESSED / "kaggle_test_clean.parquet")
split.to_csv(PROCESSED / "split.csv")

for f in sorted(PROCESSED.iterdir()):
    print(f"{f.name:35s} {f.stat().st_size / 1e6:6.1f} MB")

kaggle_test_clean.parquet              4.0 MB
labeled_clean.parquet                 20.3 MB
split.csv                              4.2 MB


Check: reading the files back gives exactly the same data.
This is also how the model notebooks load the data.

In [18]:
data = pd.read_parquet(PROCESSED / "labeled_clean.parquet")
split_loaded = pd.read_csv(PROCESSED / "split.csv", index_col="SK_ID_CURR")["split"]

pd.testing.assert_frame_equal(data, labeled_clean)
pd.testing.assert_series_equal(split_loaded, split)

train = data[split_loaded == "train"]
test = data[split_loaded == "test"]
train.shape, test.shape

((246008, 87), (61503, 87))

## 10. Summary

- **Placeholders:** 55,374 values `DAYS_EMPLOYED = 365243` became `NaN` and are marked in the new column `NOT_EMPLOYED`. 4 `CODE_GENDER` and 2 `NAME_FAMILY_STATUS` values became `NaN`.
- **Time columns:** the five `DAYS_*` columns are now `YEARS_*`. Age ranges from 20.5 to 69.1 years, employment from 0 to 49 years.
- **Ratios:** three ratio features added, without infinite values. They are missing for the 12 clients without `AMT_ANNUITY`.
- **Removed:** 38 columns, 10 near-constant and 28 duplicate building columns.
- **Result:** 86 features (70 numeric, 16 categorical) for 307,511 labeled (`labeled_clean`) and 48,744 Kaggle test clients (`kaggle_test_clean`).
- **Split:** 246,008 clients in `train` and 61,503 in `test`, both with 8.07% defaults (seed 69).

**No data leakage.** This notebook only cleans and reformats the data: it replaces placeholder values, converts days into years, adds ratios of existing columns and removes columns. It learns nothing from the data: no statistics are computed and no values are imputed. Every row is transformed on its own, so a client's cleaned values are the same whether the cleaning runs before or after the split. That is why the cleaning does not need the split. `labeled` and `kaggle_test` are cleaned separately with the same function.